# CPMK 3 — Demo Langsung

Notebook untuk sesi demo 10 menit: **inferensi foto baru**, **ganti threshold**, **AP per kelas**.

**Persiapan di Kaggle (lakukan SEBELUM sesi):** GPU T4, Internet On, Secrets `ROBOFLOW_API_KEY`, lalu *Add Input*:
1. Notebook Output `CPMK3_training` (bobot 9 run).
2. Dataset test custom (yang dipakai notebook evaluasi).
3. (Opsional) Notebook Output `CPMK3_eval_custom` → prediksi test custom tersimpan, jadi Bagian D3 tidak perlu inferensi ulang.

Jalankan semua sel sampai Bagian D1 sebelum sesi dimulai (download data + memuat model butuh beberapa menit).
Saat sesi: upload foto baru (panel kanan → *Upload* ke `/kaggle/working`, atau *Add Input*), lalu jalankan Bagian D2.

## 1. Setup

In [ ]:
!nvidia-smi
!pip -q install ultralytics roboflow pycocotools

In [ ]:
import os, sys, json, time, math, random, csv, glob, datetime, contextlib, io, copy
from collections import defaultdict
import numpy as np, cv2, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageOps
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils

import gc, traceback
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')

# ===================== KONFIGURASI UTAMA =====================
NRP_LAST_DIGIT = 3          # digit 3 -> variabel: jumlah data latih (25% / 50% / 100%)
FRACTIONS      = [0.25, 0.5, 1.0]                     # variabel eksperimen (NRP ...3)
MODELS_TO_RUN  = ['yolo', 'maskrcnn', 'mask2former']  # bisa dikurangi kalau mau run sebagian
QUICK_TEST     = False      # True = uji coba cepat (budget 2 menit/run) untuk cek error sebelum commit
IMG_SIZE       = 640        # sisi terpanjang input, sama untuk semua model
SEED           = 42
TIME_BUDGET_H  = 0.66       # batas waktu training PER RUN (jam). 3 fraksi x 0.66 jam ≈ 2 jam per model
MAX_EPOCHS     = 100        # batas atas; biasanya yang menghentikan training adalah TIME_BUDGET_H
if QUICK_TEST: TIME_BUDGET_H, MAX_EPOCHS = 0.03, 2
TRAIN_FRACTION = FRACTIONS[-1]   # diatur otomatis oleh set_fraction()

RF_WORKSPACE, RF_PROJECT, RF_VERSION = "panopticindoor", "panoptic-indoor-segmentation", 5

if IN_KAGGLE:   # /kaggle/working = output tersimpan (maks 20 GB); data mentah taruh di /tmp agar tidak ikut tersimpan
    OUT_ROOT, DATA_ROOT = "/kaggle/working/CPMK3", "/tmp/cpmk3_data"
elif IN_COLAB:
    OUT_ROOT, DATA_ROOT = "/content/drive/MyDrive/CPMK3", "/content/data"
else:
    OUT_ROOT, DATA_ROOT = os.path.abspath("cpmk3_out"), os.path.abspath("cpmk3_data")
if QUICK_TEST: OUT_ROOT += "_quicktest"
# =============================================================

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print('device:', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '', '| torch', torch.__version__)
print('output ->', OUT_ROOT)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
for d in [OUT_ROOT, f"{OUT_ROOT}/runs", f"{OUT_ROOT}/results", f"{OUT_ROOT}/preds", DATA_ROOT]:
    os.makedirs(d, exist_ok=True)

### 1.1 Logbook & pencatat waktu GPU (bukti proses, poin 4)
- `gpu_time_log.csv`: tiap run training otomatis tercatat (GPU, mulai, selesai, menit, peak memory, status).
- `LOGBOOK.md`: catat kendala dan keputusan dengan `log_note("...")`. Contoh: `log_note("YOLO OOM di batch 32, turun ke 16")`.

In [ ]:
GPU_LOG = f"{OUT_ROOT}/gpu_time_log.csv"
LOGBOOK = f"{OUT_ROOT}/LOGBOOK.md"

def log_note(text):
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    with open(LOGBOOK, 'a') as f:
        f.write(f"- **{ts}** — {text}\n")
    print('logbook +', text)

class GPUTimer:
    # Dipakai sebagai: with GPUTimer(run, model) as t: ...  -> t.minutes
    def __init__(self, run, model, note=''):
        self.run, self.model, self.note, self.minutes = run, model, note, None
    def __enter__(self):
        self.start, self.t0 = datetime.datetime.now(), time.time()
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        return self
    def __exit__(self, exc_type, exc, tb):
        self.minutes = (time.time() - self.t0) / 60
        cuda = torch.cuda.is_available()
        gpu  = torch.cuda.get_device_name(0) if cuda else 'cpu'
        peak = torch.cuda.max_memory_allocated() / 1e9 if cuda else 0.0
        status = 'ok' if exc_type is None else f'stopped: {exc_type.__name__}'
        new = not os.path.exists(GPU_LOG)
        with open(GPU_LOG, 'a', newline='') as f:
            w = csv.writer(f)
            if new: w.writerow(['run','model','gpu','start','end','minutes','peak_mem_gb','status','note'])
            w.writerow([self.run, self.model, gpu, self.start.isoformat(timespec='seconds'),
                        datetime.datetime.now().isoformat(timespec='seconds'),
                        f"{self.minutes:.2f}", f"{peak:.2f}", status, self.note])
        print(f"⏱  {self.run}: {self.minutes:.1f} menit di {gpu} (peak {peak:.1f} GB) — {status}")
        return False   # error tetap dilempar

def gpu_minutes(run):
    # total menit GPU sebuah run dari log (berguna kalau sesi Colab sudah restart)
    if not os.path.exists(GPU_LOG): return None
    df = pd.read_csv(GPU_LOG)
    return float(df[df.run == run].minutes.sum()) or None

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 2. Dataset

### 2.1 Download dari Roboflow (format COCO segmentation)
API key dibaca dari Kaggle Secrets / Colab Secrets dengan nama `ROBOFLOW_API_KEY`.

In [ ]:
COCO_DIR = f"{DATA_ROOT}/coco"
if not os.path.exists(f"{COCO_DIR}/train/_annotations.coco.json"):
    from roboflow import Roboflow
    RF_KEY = os.environ.get('ROBOFLOW_API_KEY')
    if not RF_KEY and IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        RF_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    if not RF_KEY and IN_COLAB:
        from google.colab import userdata
        RF_KEY = userdata.get('ROBOFLOW_API_KEY')
    if not RF_KEY:
        import getpass; RF_KEY = getpass.getpass('Roboflow API key: ')
    rf = Roboflow(api_key=RF_KEY)
    version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
    # overwrite=True: tanpa ini roboflow DIAM-DIAM melewati download kalau folder sudah ada (mis. sisa run gagal)
    try:
        version.download("coco-segmentation", location=COCO_DIR, overwrite=True)
    except Exception as e:
        print("format coco-segmentation gagal:", repr(e)[:200], "-> coba format 'coco'")
        version.download("coco", location=COCO_DIR, overwrite=True)

# cari file anotasi di mana pun di dalam COCO_DIR (antisipasi struktur folder berbeda)
ann_files = sorted(glob.glob(f"{COCO_DIR}/**/_annotations.coco.json", recursive=True))
IMG_DIRS = {os.path.basename(os.path.dirname(p)).lower(): os.path.dirname(p) for p in ann_files}
if 'val' in IMG_DIRS and 'valid' not in IMG_DIRS: IMG_DIRS['valid'] = IMG_DIRS.pop('val')
SPLITS = [s for s in ['train', 'valid', 'test'] if s in IMG_DIRS]
print('split tersedia:', {s: IMG_DIRS[s] for s in SPLITS})
if not SPLITS:
    print("Isi COCO_DIR (untuk diagnosis):")
    for root, dirs, files in os.walk(COCO_DIR):
        print(' ', root, '|', len(files), 'file', files[:3])
    raise RuntimeError("File _annotations.coco.json tidak ditemukan. Cek output di atas / API key / akses internet.")
assert 'train' in SPLITS and 'test' in SPLITS, f"Split kurang: {SPLITS} — catat di logbook dan buat split dari train/valid."

### 2.2 Bersihkan anotasi
Export COCO dari Roboflow biasanya punya satu kategori "induk" (superkategori) yang tidak dipakai anotasi mana pun, jadi kategori itu dibuang. Kelas = kategori yang benar-benar muncul di train. Anotasi tanpa polygon valid (kurang dari 3 titik) juga dibuang. Hasilnya ditulis ke `clean/{split}.json`, dan semua model membaca file ini.

In [ ]:
CLEAN_DIR = f"{DATA_ROOT}/clean"; os.makedirs(CLEAN_DIR, exist_ok=True)
raw = {s: json.load(open(f"{IMG_DIRS[s]}/_annotations.coco.json")) for s in SPLITS}

cats_by_id  = {c['id']: c for c in raw['train']['categories']}
name2id     = {c['name']: c['id'] for c in raw['train']['categories']}
CAT_IDS     = sorted({a['category_id'] for a in raw['train']['annotations']})
CLASS_NAMES = [cats_by_id[i]['name'] for i in CAT_IDS]
NUM_CLASSES = len(CAT_IDS)
CATID2IDX   = {cid: i for i, cid in enumerate(CAT_IDS)}   # id COCO -> indeks 0..K-1
print(f"{NUM_CLASSES} kelas:", CLASS_NAMES)
print("kategori dibuang (tanpa anotasi di train):", [c['name'] for c in raw['train']['categories'] if c['id'] not in CATID2IDX])

def clean_split(js):
    local = {c['id']: c['name'] for c in js['categories']}
    anns, dropped = [], defaultdict(int)
    for a in js['annotations']:
        cid = name2id.get(local.get(a['category_id']))          # samakan id antar split lewat nama
        if cid not in CATID2IDX: dropped['kelas tidak ada di train'] += 1; continue
        seg = a.get('segmentation')
        if isinstance(seg, list):
            seg = [p for p in seg if len(p) >= 6]
            if not seg: dropped['polygon kosong/<3 titik'] += 1; continue
        elif not seg: dropped['tanpa segmentasi'] += 1; continue
        a = dict(a, category_id=cid, segmentation=seg)
        a.setdefault('iscrowd', 0)
        a.setdefault('area', float(a['bbox'][2] * a['bbox'][3]))
        anns.append(a)
    out = {'info': js.get('info', {}), 'images': js['images'], 'annotations': anns,
           'categories': [cats_by_id[i] for i in CAT_IDS]}
    return out, dict(dropped)

for s in SPLITS:
    out, dropped = clean_split(raw[s])
    json.dump(out, open(f"{CLEAN_DIR}/{s}.json", 'w'))
    print(f"{s:6s}: {len(out['images']):5d} gambar, {len(out['annotations']):6d} instance, dibuang: {dropped or '-'}")

### 2.3 EDA singkat
Jumlah instance per kelas dan proporsi objek kecil/sedang/besar (definisi COCO: kecil < 32², besar > 96² piksel). Bagian ini berguna nanti untuk menjelaskan kenapa AP tiap kelas berbeda.

In [ ]:
rows = []
for s in SPLITS:
    js = json.load(open(f"{CLEAN_DIR}/{s}.json"))
    for a in js['annotations']:
        area = a['area']
        rows.append(dict(split=s, cls=CLASS_NAMES[CATID2IDX[a['category_id']]],
                         size='small' if area < 32**2 else ('large' if area > 96**2 else 'medium')))
eda = pd.DataFrame(rows)
cls_table = eda.pivot_table(index='cls', columns='split', aggfunc='size', fill_value=0)
cls_table = cls_table[[s for s in SPLITS if s in cls_table.columns]].sort_values('train', ascending=False)
display(cls_table)
display(pd.crosstab(eda.cls, eda['size'], normalize='index').round(2).loc[cls_table.index])

ax = cls_table['train'].plot.barh(figsize=(7, max(3, 0.3 * NUM_CLASSES)), title='Instance per kelas (train)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
def load_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)   # samakan orientasi dengan cv2.imread (dipakai Ultralytics)
    return img.convert('RGB')

PALETTE = np.random.RandomState(0).randint(40, 255, (256, 3))

def draw_instances(img, masks, labels, scores=None, ax=None, title=''):
    # img: PIL/np RGB; masks: list/array (N,H,W) bool; labels: indeks 0..K-1
    im = np.array(img).astype(np.float32)
    for i, (m, l) in enumerate(zip(masks, labels)):
        im[m] = 0.5 * im[m] + 0.5 * PALETTE[i % 256]
    ax = ax or plt.gca()
    ax.imshow(im.astype(np.uint8)); ax.set_title(title, fontsize=9); ax.axis('off')
    for i, (m, l) in enumerate(zip(masks, labels)):
        ys, xs = np.nonzero(m)
        if len(xs):
            txt = CLASS_NAMES[l] + (f" {scores[i]:.2f}" if scores is not None else '')
            ax.text(xs.mean(), ys.mean(), txt, fontsize=6, color='white', bbox=dict(facecolor='black', alpha=0.5, pad=1))

def show_gt(split, n=4, seed=0):
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(f"{CLEAN_DIR}/{split}.json")
    ids = random.Random(seed).sample(sorted(coco.imgs), min(n, len(coco.imgs)))
    fig, axs = plt.subplots(1, len(ids), figsize=(5 * len(ids), 5))
    for ax, i in zip(np.atleast_1d(axs), ids):
        info = coco.imgs[i]; anns = coco.loadAnns(coco.getAnnIds(imgIds=i, iscrowd=False))
        draw_instances(load_image(os.path.join(IMG_DIRS[split], info['file_name'])),
                       [coco.annToMask(a).astype(bool) for a in anns],
                       [CATID2IDX[a['category_id']] for a in anns], ax=ax, title=info['file_name'][:30])
    plt.tight_layout(); plt.show()

show_gt('train')

### 2.4 Subset data latih (variabel NRP)
Gambar train yang punya anotasi diacak **sekali** dengan `SEED`, lalu diambil prefiks sebesar `TRAIN_FRACTION`. Karena urutannya sama, subset 25% ⊂ 50% ⊂ 100%. Validasi dan test **tidak** dikurangi.

Perhatikan output distribusi kelas: di 25%, kelas langka bisa tinggal sedikit sekali contohnya. Ini bahan analisis yang bagus.

In [ ]:
def make_train_subset(frac, seed=SEED):
    js = json.load(open(f"{CLEAN_DIR}/train.json"))
    has_ann = sorted({a['image_id'] for a in js['annotations'] if not a.get('iscrowd', 0)})
    order = has_ann[:]; random.Random(seed).shuffle(order)
    keep = set(order[:max(1, int(round(frac * len(order))))])
    sub = dict(js, images=[im for im in js['images'] if im['id'] in keep],
               annotations=[a for a in js['annotations'] if a['image_id'] in keep])
    path = f"{CLEAN_DIR}/train_f{int(round(frac * 100))}.json"
    json.dump(sub, open(path, 'w'))
    cnt = pd.Series([CLASS_NAMES[CATID2IDX[a['category_id']]] for a in sub['annotations']]).value_counts()
    print(f"subset {frac:.0%}: {len(sub['images'])} gambar, {len(sub['annotations'])} instance -> {path}")
    print("kelas paling sedikit:", cnt.tail(5).to_dict())
    return path

def set_fraction(frac):
    # ganti fraksi aktif: semua fungsi run_* membaca TRAIN_FRACTION & TRAIN_JSON global
    global TRAIN_FRACTION, TRAIN_JSON
    TRAIN_FRACTION, TRAIN_JSON = frac, make_train_subset(frac)

for f in FRACTIONS: set_fraction(f)      # tampilkan distribusi kelas tiap subset
VAL_JSON   = f"{CLEAN_DIR}/valid.json"
TEST_JSON  = f"{CLEAN_DIR}/test.json"

## 3. Evaluasi bersama (COCO mask mAP)
Semua model mengeluarkan prediksi dalam format hasil COCO (`image_id, category_id, segmentation RLE, score`), lalu dinilai dengan fungsi yang sama:
- **AP** = mAP@[.50:.95] (metrik utama), **AP50**, **AP75**, **APs/APm/APl** (per ukuran objek)
- **AP per kelas**, dipakai untuk analisis kelas sulit dan analisis kegagalan

Prediksi disimpan ke Drive (`preds/`), jadi visualisasi kegagalan nanti bisa dilakukan tanpa inferensi ulang.

In [ ]:
import torchvision.transforms.functional as TF

def load_instances(coco, img_id):
    # -> masks (N,H,W) uint8, labels 0..K-1; buang mask kosong
    masks, labels = [], []
    for a in coco.loadAnns(coco.getAnnIds(imgIds=img_id, iscrowd=False)):
        m = coco.annToMask(a)
        if m.any(): masks.append(m); labels.append(CATID2IDX[a['category_id']])
    H, W = coco.imgs[img_id]['height'], coco.imgs[img_id]['width']
    return (np.stack(masks) if masks else np.zeros((0, H, W), np.uint8)), np.array(labels, np.int64)

def mask_to_rle(m):
    rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))
    rle['counts'] = rle['counts'].decode('ascii')
    return rle

METRIC_NAMES = ['AP', 'AP50', 'AP75', 'APs', 'APm', 'APl']

def coco_eval_segm(gt_json, results, verbose=True):
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(gt_json)
    if len(results) == 0:
        return {**{k: 0.0 for k in METRIC_NAMES}, 'per_class': {n: 0.0 for n in CLASS_NAMES}}
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        dt = gt.loadRes(copy.deepcopy(results))     # loadRes memodifikasi list (area numpy) -> salin
        E = COCOeval(gt, dt, 'segm'); E.evaluate(); E.accumulate(); E.summarize()
    if verbose: print('\n'.join(l for l in buf.getvalue().splitlines() if l.startswith(' Average')))
    out = {k: float(E.stats[i]) for i, k in enumerate(METRIC_NAMES)}
    prec = E.eval['precision']              # [IoU, recall, kelas, area, maxDet]
    out['per_class'] = {}
    for k, cid in enumerate(E.params.catIds):
        p = prec[:, :, k, 0, -1]; p = p[p > -1]
        out['per_class'][cats_by_id[cid]['name']] = float(p.mean()) if p.size else float('nan')
    return out

def run_name(model_key):
    return f"{model_key}_f{int(round(TRAIN_FRACTION * 100))}_s{IMG_SIZE}"

def already_done(run, split='test'):
    done = os.path.exists(f"{OUT_ROOT}/results/{run}__{split}.json")
    if done: print(f"⏭  {run} sudah ada hasilnya, dilewati (hapus file results-nya untuk mengulang)")
    return done

def save_run(run, model_name, family, eval_split, metrics, preds, extra=None):
    rec = dict(run=run, model=model_name, family=family, eval_split=eval_split,
               train_fraction=TRAIN_FRACTION, img_size=IMG_SIZE, seed=SEED,
               gpu_minutes=gpu_minutes(run), n_preds=len(preds),
               timestamp=datetime.datetime.now().isoformat(timespec='seconds'),
               **{k: v for k, v in metrics.items() if k != 'per_class'},
               per_class=metrics['per_class'], **(extra or {}))
    json.dump(rec, open(f"{OUT_ROOT}/results/{run}__{eval_split}.json", 'w'), indent=2)
    json.dump(preds, open(f"{OUT_ROOT}/preds/{run}__{eval_split}.json", 'w'))
    print(f"✅ {run} @ {eval_split}: AP={rec['AP']:.4f} AP50={rec['AP50']:.4f}  (disimpan)")
    return rec

def show_preds(run, eval_split, gt_json, img_dir, n=3, score_thr=0.5, img_ids=None, seed=0):
    # GT (kiri) vs prediksi (kanan) dari file prediksi tersimpan
    preds = json.load(open(f"{OUT_ROOT}/preds/{run}__{eval_split}.json"))
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    img_ids = img_ids or random.Random(seed).sample(sorted(coco.imgs), min(n, len(coco.imgs)))
    by_img = defaultdict(list)
    for p in preds:
        if p['score'] >= score_thr: by_img[p['image_id']].append(p)
    fig, axs = plt.subplots(len(img_ids), 2, figsize=(11, 5 * len(img_ids)), squeeze=False)
    for row, i in zip(axs, img_ids):
        info = coco.imgs[i]; img = load_image(os.path.join(img_dir, info['file_name']))
        anns = coco.loadAnns(coco.getAnnIds(imgIds=i, iscrowd=False))
        draw_instances(img, [coco.annToMask(a).astype(bool) for a in anns],
                       [CATID2IDX[a['category_id']] for a in anns], ax=row[0], title=f"GT {info['file_name'][:25]}")
        ps = by_img[i]
        draw_instances(img, [mask_utils.decode(p['segmentation']).astype(bool) for p in ps],
                       [CATID2IDX[p['category_id']] for p in ps], [p['score'] for p in ps],
                       ax=row[1], title=f"{run} (score≥{score_thr})")
    plt.tight_layout(); plt.show()

### 3.1 Training loop generik (untuk Mask R-CNN & Mask2Former)
YOLO memakai trainer Ultralytics. Dua model lain memakai loop yang sama:
- **LR schedule**: warmup linear, lalu cosine. Progres dihitung dari `max(iterasi/total, waktu/budget)`, jadi LR tetap turun ke minimum walaupun training dihentikan oleh batas waktu, bukan oleh jumlah epoch. Ultralytics juga melakukan hal serupa saat `time=` dipakai.
- Validasi (COCO segm AP di `valid`) tiap `val_every` epoch, lalu simpan `best.pt` dan `last.pt` ke Drive.
- Log per epoch ke `results.csv` (bukti proses, poin 4).
- Step dengan loss NaN/inf dilewati dan dihitung; kalau banyak, matikan AMP.

In [ ]:
def train_loop(run, model, dl_train, step_fn, optimizer, eval_fn, budget_h=TIME_BUDGET_H,
               max_epochs=MAX_EPOCHS, val_every=1, amp=USE_AMP, clip=None, warmup_iters=300, min_lr_ratio=0.01):
    out_dir = f"{OUT_ROOT}/runs/{run}"; os.makedirs(out_dir, exist_ok=True)
    csv_path = f"{out_dir}/results.csv"
    if os.path.exists(csv_path): os.remove(csv_path)
    base_lrs = [g['lr'] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    t0, budget, total = time.time(), budget_h * 3600, max_epochs * len(dl_train)
    it, best_ap, skipped, stop = 0, -1.0, 0, False
    for ep in range(1, max_epochs + 1):
        model.train(); sums, n = defaultdict(float), 0
        for batch in dl_train:
            prog = max(it / total, (time.time() - t0) / budget)
            if prog >= 1: stop = True; break
            f = min(1.0, (it + 1) / warmup_iters) * (min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * prog)))
            for g, b in zip(optimizer.param_groups, base_lrs): g['lr'] = b * f
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=amp):
                loss, logs = step_fn(model, batch)
            it += 1
            if not torch.isfinite(loss):
                skipped += 1; optimizer.zero_grad(set_to_none=True); continue
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if clip:
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer); scaler.update()
            for k, v in logs.items(): sums[k] += v
            n += 1
            if it % 50 == 0:
                print(f"  ep{ep} it{it} loss={logs.get('loss', loss.item()):.3f} lr={optimizer.param_groups[0]['lr']:.2e} "
                      f"t={(time.time()-t0)/60:.1f}m", flush=True)
        if ep == max_epochs: stop = True
        row = {'epoch': ep, 'iters': it, 'minutes': round((time.time() - t0) / 60, 2),
               'lr': optimizer.param_groups[0]['lr'], 'skipped_steps': skipped,
               **{f"train/{k}": v / max(n, 1) for k, v in sums.items()},
               **{f"val/{k}": np.nan for k in METRIC_NAMES}}       # kolom tetap sama tiap baris
        if ep % val_every == 0 or stop:
            model.eval(); m = eval_fn(model)
            row.update({f"val/{k}": m[k] for k in METRIC_NAMES})
            if m['AP'] > best_ap:
                best_ap = m['AP']; torch.save(model.state_dict(), f"{out_dir}/best.pt")
            print(f"[ep {ep}] {row['minutes']} menit | val AP={m['AP']:.4f} AP50={m['AP50']:.4f} | best={best_ap:.4f}")
        pd.DataFrame([row]).to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
        if stop: break
    torch.save(model.state_dict(), f"{out_dir}/last.pt")
    if skipped > 0.1 * max(it, 1): print(f"⚠️ {skipped}/{it} step NaN — pertimbangkan amp=False")
    return f"{out_dir}/best.pt"

## 4. Model A — YOLO11s-seg (one-stage, CNN)

**Cara kerja singkat**: backbone CNN dan neck FPN/PAN menghasilkan prediksi padat di tiap grid untuk box, kelas, dan 32 koefisien mask. Ada cabang *Proto* yang menghasilkan 32 prototype mask berukuran **1/4 input** (160×160 untuk 640). Mask tiap objek = sigmoid(koefisien · prototype), di-crop ke box, lalu diperbesar. Duplikat dihapus dengan NMS.

Konsekuensi yang perlu diingat untuk analisis kegagalan:
- detail tipis/kecil < 4 piksel input hilang karena resolusi prototype;
- mask "terpotong lurus" kalau box-nya kekecilan (karena crop by box);
- objek bertumpuk dengan kelas sama bisa tertekan NMS.

Konversi: COCO → format label YOLO (polygon ternormalisasi). Objek dengan beberapa polygon (misal terpotong oleh objek lain) digabung dengan `merge_multi_segment` dari Ultralytics, sama seperti konverter resmi mereka.

In [ ]:
from ultralytics import YOLO
import yaml
try:
    from ultralytics.data.converter import merge_multi_segment
except Exception:
    merge_multi_segment = None

YOLO_DIR = f"{DATA_ROOT}/yolo"

def ann_to_polys(coco, a):
    seg = a['segmentation']
    if isinstance(seg, list):
        return [np.array(p, dtype=np.float32).reshape(-1, 2) for p in seg if len(p) >= 6]
    m = coco.annToMask(a)                                   # RLE -> kontur
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c.reshape(-1, 2).astype(np.float32) for c in cnts if len(c) >= 3]

def write_yolo_split(split):
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(f"{CLEAN_DIR}/{split}.json")
    img_out, lbl_out = f"{YOLO_DIR}/images/{split}", f"{YOLO_DIR}/labels/{split}"
    os.makedirs(img_out, exist_ok=True); os.makedirs(lbl_out, exist_ok=True)
    n_multi = 0
    for img in coco.dataset['images']:
        dst = os.path.join(img_out, img['file_name'])
        if not os.path.lexists(dst): os.symlink(os.path.join(IMG_DIRS[split], img['file_name']), dst)
        W, H = img['width'], img['height']; lines = []
        for a in coco.loadAnns(coco.getAnnIds(imgIds=img['id'])):
            if a.get('iscrowd', 0): continue
            polys = ann_to_polys(coco, a)
            if not polys: continue
            if len(polys) > 1:
                n_multi += 1
                if merge_multi_segment is not None:
                    pts = np.concatenate(merge_multi_segment([p.reshape(-1).tolist() for p in polys]), 0)
                else:
                    pts = max(polys, key=cv2.contourArea)
            else:
                pts = polys[0]
            pts = (pts / np.array([W, H], dtype=np.float32)).clip(0, 1)
            lines.append(f"{CATID2IDX[a['category_id']]} " + " ".join(f"{v:.6f}" for v in pts.reshape(-1)))
        with open(os.path.join(lbl_out, os.path.splitext(img['file_name'])[0] + '.txt'), 'w') as f:
            f.write("\n".join(lines))
    print(f"{split}: {len(coco.imgs)} gambar, {n_multi} instance multi-polygon digabung")

for s in SPLITS:
    write_yolo_split(s)

def yolo_data_yaml():
    # daftar gambar subset train aktif (path absolut) + data.yaml untuk fraksi tsb
    sub = json.load(open(TRAIN_JSON))
    tag = f"f{int(round(TRAIN_FRACTION * 100))}"
    train_txt = f"{YOLO_DIR}/train_{tag}.txt"
    with open(train_txt, 'w') as f:
        f.write("\n".join(f"{YOLO_DIR}/images/train/{im['file_name']}" for im in sub['images']))
    path = f"{YOLO_DIR}/data_{tag}.yaml"
    yaml.safe_dump({'path': YOLO_DIR, 'train': train_txt, 'val': f"{YOLO_DIR}/images/valid",
                    'test': f"{YOLO_DIR}/images/test", 'names': dict(enumerate(CLASS_NAMES))}, open(path, 'w'))
    return path

In [ ]:
YOLO_WEIGHTS = 'yolo11s-seg.pt'      # pretrained COCO

@torch.no_grad()
def predict_yolo(weights, gt_json, img_dir):
    model = YOLO(weights)
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        r = model.predict(os.path.join(img_dir, img['file_name']), imgsz=IMG_SIZE, conf=0.001, iou=0.7,
                          max_det=100, retina_masks=True, verbose=False,
                          device=0 if DEVICE.type == 'cuda' else 'cpu')[0]
        if r.masks is None or len(r.boxes) == 0: continue
        H, W = img['height'], img['width']
        for m, c, s in zip(r.masks.data.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            if m.shape != (H, W): m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
            res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(c)],
                        'segmentation': mask_to_rle(m > 0.5), 'score': float(s)})
    return res

def run_yolo():
    run = run_name('yolo11s-seg')
    if already_done(run): return
    seed_everything()
    with GPUTimer(run, 'YOLO11s-seg', note=f"frac={TRAIN_FRACTION}"):
        YOLO(YOLO_WEIGHTS).train(
            data=yolo_data_yaml(), imgsz=IMG_SIZE, epochs=MAX_EPOCHS, time=TIME_BUDGET_H,   # time= override epochs
            batch=16, seed=SEED, workers=2, amp=USE_AMP, device=0 if DEVICE.type == 'cuda' else 'cpu',
            project=f"{OUT_ROOT}/runs", name=run, exist_ok=True, plots=True)
    # log asli Ultralytics: {OUT_ROOT}/runs/{run}/results.csv
    preds = predict_yolo(f"{OUT_ROOT}/runs/{run}/weights/best.pt", TEST_JSON, IMG_DIRS['test'])
    save_run(run, 'YOLO11s-seg', 'one-stage CNN', 'test', coco_eval_segm(TEST_JSON, preds), preds)

## 5. Model B — Mask R-CNN R50-FPN v2 (two-stage, CNN)

**Cara kerja singkat**:
1. **Stage 1 (RPN)**: mengusulkan ~1000 region kandidat.
2. **Stage 2**: RoIAlign memotong fitur tiap region, lalu head box/kelas dan **mask head memprediksi grid 28×28** biner per kelas. Grid ini di-*resize* ke ukuran box di gambar asli.

Konsekuensi untuk analisis kegagalan:
- objek besar (kasur, lemari, dinding) diwakili hanya 28×28 sel, jadi tepinya kasar/"bocor" setelah di-upsample;
- kalau box salah (terlalu kecil), mask ikut terpotong;
- objek memanjang/tipis kehilangan detail.

Implementasi: `torchvision` (tanpa Detectron2, supaya instalasi di Colab tidak bermasalah). Input di-resize oleh model sehingga sisi terpanjang = `IMG_SIZE`.

In [ ]:
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

class MaskRCNNDataset(torch.utils.data.Dataset):
    def __init__(self, ann_json, img_dir, train=False):
        with contextlib.redirect_stdout(io.StringIO()):
            self.coco = COCO(ann_json)
        self.img_dir, self.train = img_dir, train
        self.ids = sorted(self.coco.imgs)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img_id = self.ids[i]; info = self.coco.imgs[img_id]
        img = TF.to_tensor(load_image(os.path.join(self.img_dir, info['file_name'])))
        masks, labels = load_instances(self.coco, img_id)
        masks = torch.from_numpy(masks)
        if self.train and random.random() < 0.5:                 # augmentasi: flip horizontal
            img, masks = img.flip(-1), masks.flip(-1)
        boxes = []
        for m in masks:                                          # box diturunkan dari mask -> konsisten
            ys, xs = torch.nonzero(m, as_tuple=True)
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        target = {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                  'labels': torch.from_numpy(labels) + 1,          # 0 = background di torchvision
                  'masks': masks}
        return img, target

def collate_tuple(batch): return tuple(zip(*batch))

def build_maskrcnn(num_classes=NUM_CLASSES, pretrained=True):
    w = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn_v2(weights=w, min_size=IMG_SIZE, max_size=IMG_SIZE)   # sisi terpanjang = IMG_SIZE
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes + 1)
    return model.to(DEVICE)

def mrcnn_step(model, batch):
    imgs, tgts = batch
    imgs = [i.to(DEVICE, non_blocking=True) for i in imgs]
    tgts = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in tgts]
    ld = model(imgs, tgts)
    loss = sum(ld.values())
    return loss, {'loss': loss.item(), **{k: v.item() for k, v in ld.items()}}

@torch.no_grad()
def predict_maskrcnn(model, gt_json, img_dir):
    model.eval(); model.roi_heads.score_thresh = 0.001; model.roi_heads.detections_per_img = 100
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        x = TF.to_tensor(load_image(os.path.join(img_dir, img['file_name']))).to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model([x])[0]
        masks = (out['masks'][:, 0] > 0.5).cpu().numpy()
        for m, l, s in zip(masks, out['labels'].cpu().numpy(), out['scores'].float().cpu().numpy()):
            res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(l) - 1],
                        'segmentation': mask_to_rle(m), 'score': float(s)})
    return res

In [ ]:
MRCNN_BS, MRCNN_LR = 4, 0.01      # LR SGD untuk batch 4 (skala linear dari resep torchvision)

def run_mrcnn():
    run = run_name('maskrcnn-r50fpnv2')
    if already_done(run): return
    seed_everything()
    model = build_maskrcnn()
    dl = torch.utils.data.DataLoader(MaskRCNNDataset(TRAIN_JSON, IMG_DIRS['train'], train=True), batch_size=MRCNN_BS,
                                     shuffle=True, num_workers=2, collate_fn=collate_tuple, pin_memory=True, drop_last=True)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=MRCNN_LR, momentum=0.9, weight_decay=1e-4)
    val_fn = lambda m: coco_eval_segm(VAL_JSON, predict_maskrcnn(m, VAL_JSON, IMG_DIRS['valid']), verbose=False)
    with GPUTimer(run, 'Mask R-CNN R50-FPN v2', note=f"frac={TRAIN_FRACTION}"):
        best = train_loop(run, model, dl, mrcnn_step, opt, val_fn, clip=10.0)
    model.load_state_dict(torch.load(best, map_location=DEVICE))
    preds = predict_maskrcnn(model, TEST_JSON, IMG_DIRS['test'])
    save_run(run, 'Mask R-CNN R50-FPN v2', 'two-stage CNN', 'test', coco_eval_segm(TEST_JSON, preds), preds)

## 6. Model C — Mask2Former Swin-Tiny (transformer)

**Cara kerja singkat**:
1. Backbone **Swin Transformer** dan *pixel decoder* (multi-scale deformable attention) menghasilkan fitur per piksel di resolusi 1/4.
2. **Transformer decoder** dengan 100 *object query*. Tiap layer memakai **masked attention**: query hanya memperhatikan piksel di dalam mask prediksinya sendiri dari layer sebelumnya.
3. Tiap query mengeluarkan (kelas, mask). Training memakai **Hungarian matching** antara query dan GT, jadi tidak perlu NMS dan tidak ada box sama sekali.

Konsekuensi untuk analisis kegagalan:
- mask di 1/4 resolusi (mirip YOLO), tapi **tidak dibatasi box**, sehingga tidak ada efek "terpotong lurus", meski mask bisa "bocor" ke area dengan tekstur mirip;
- jumlah objek maksimum = 100 query;
- butuh data/epoch lebih banyak untuk konvergen. Ini relevan dengan variabel fraksi data.

**Preprocessing (ditulis manual)**: resize sisi terpanjang ke `IMG_SIZE`, pad ke persegi `IMG_SIZE×IMG_SIZE` (+`pixel_mask`), normalisasi ImageNet. Post-processing juga ditulis manual mengikuti paper: skor = prob kelas × rata-rata prob mask di area mask. Area padding dipotong sebelum mask dikembalikan ke resolusi asli.

Jika loss sering NaN dengan AMP: set `M2F_AMP = False` (lebih lambat ±2×) dan catat di logbook. Jika OOM: `M2F_BS = 2`.

In [ ]:
from transformers import Mask2FormerForUniversalSegmentation

M2F_CKPT = "facebook/mask2former-swin-tiny-coco-instance"
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def m2f_preprocess(img, S=IMG_SIZE):
    W, H = img.size; sc = S / max(H, W)
    nh, nw = max(1, round(H * sc)), max(1, round(W * sc))
    x = (TF.to_tensor(img.resize((nw, nh), Image.BILINEAR)) - IMNET_MEAN) / IMNET_STD
    pv = torch.zeros(3, S, S); pv[:, :nh, :nw] = x
    pm = torch.zeros(S, S, dtype=torch.long); pm[:nh, :nw] = 1
    return pv, pm, (nh, nw)

class M2FDataset(torch.utils.data.Dataset):
    def __init__(self, ann_json, img_dir, train=False):
        with contextlib.redirect_stdout(io.StringIO()):
            self.coco = COCO(ann_json)
        self.img_dir, self.train, self.ids = img_dir, train, sorted(self.coco.imgs)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img_id = self.ids[i]; info = self.coco.imgs[img_id]
        img = load_image(os.path.join(self.img_dir, info['file_name']))
        masks, labels = load_instances(self.coco, img_id)
        if self.train and random.random() < 0.5:
            img = ImageOps.mirror(img); masks = masks[:, :, ::-1]
        pv, pm, (nh, nw) = m2f_preprocess(img)
        ml = torch.zeros(len(masks), IMG_SIZE, IMG_SIZE)
        for k, m in enumerate(masks):
            ml[k, :nh, :nw] = torch.from_numpy(cv2.resize(np.ascontiguousarray(m), (nw, nh), interpolation=cv2.INTER_NEAREST)).float()
        keep = ml.flatten(1).sum(1) > 0                          # instance yang hilang setelah resize dibuang
        return {'pixel_values': pv, 'pixel_mask': pm, 'mask_labels': ml[keep],
                'class_labels': torch.from_numpy(labels)[keep]}

def m2f_collate(batch):
    return {'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
            'pixel_mask': torch.stack([b['pixel_mask'] for b in batch]),
            'mask_labels': [b['mask_labels'] for b in batch],
            'class_labels': [b['class_labels'] for b in batch]}

def build_m2f(pretrained=True):
    id2label = dict(enumerate(CLASS_NAMES))
    if pretrained:
        model = Mask2FormerForUniversalSegmentation.from_pretrained(
            M2F_CKPT, id2label=id2label, label2id={v: k for k, v in id2label.items()}, ignore_mismatched_sizes=True)
    else:
        from transformers import Mask2FormerConfig
        cfg = Mask2FormerConfig.from_pretrained(M2F_CKPT, id2label=id2label, label2id={v: k for k, v in id2label.items()})
        model = Mask2FormerForUniversalSegmentation(cfg)
    # bobot kelas "no object" pada loss: ukurannya berubah (81 -> K+1), set ulang secara eksplisit
    ew = torch.ones(NUM_CLASSES + 1); ew[-1] = model.config.no_object_weight
    model.criterion.empty_weight.copy_(ew)
    return model.to(DEVICE)

def m2f_step(model, batch):
    out = model(pixel_values=batch['pixel_values'].to(DEVICE, non_blocking=True),
                pixel_mask=batch['pixel_mask'].to(DEVICE, non_blocking=True),
                mask_labels=[m.to(DEVICE) for m in batch['mask_labels']],
                class_labels=[c.to(DEVICE) for c in batch['class_labels']])
    return out.loss, {'loss': out.loss.item()}

@torch.no_grad()
def m2f_postprocess(cls_logits, mask_logits, nh, nw, H, W, S=IMG_SIZE, topk=100, chunk=10):
    # cls_logits (Q,K+1), mask_logits (Q,h,w) -> list (mask bool HxW, label, score)
    # Diproses per `chunk` query: 100 mask float di resolusi asli foto besar (~20 MP) butuh ~8 GB -> OOM di T4.
    # Hasilnya identik dengan memproses semua query sekaligus.
    K = cls_logits.shape[-1] - 1
    probs = cls_logits.float().softmax(-1)[:, :-1]              # buang kelas "no object"
    scores, idx = probs.flatten().topk(min(topk, probs.numel()))
    labels, qidx = idx % K, idx // K
    masks, mscores = [], []
    for c in range(0, len(qidx), chunk):
        m = F.interpolate(mask_logits[qidx[c:c + chunk]][None].float(), size=(S, S), mode='bilinear', align_corners=False)[0]
        m = F.interpolate(m[None, :, :nh, :nw], size=(H, W), mode='bilinear', align_corners=False)[0]   # buang padding
        prob = m.sigmoid(); binm = prob > 0.5
        mscores.append((prob * binm).flatten(1).sum(1) / (binm.flatten(1).sum(1) + 1e-6))
        masks.append(binm.cpu().numpy()); del m, prob, binm
    mscore = torch.cat(mscores) if mscores else scores.new_zeros(0)
    binm = np.concatenate(masks) if masks else np.zeros((0, H, W), bool)
    return binm, labels.cpu().numpy(), (scores * mscore).cpu().numpy()

@torch.no_grad()
def predict_m2f(model, gt_json, img_dir):
    model.eval()
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        pil = load_image(os.path.join(img_dir, img['file_name']))
        pv, pm, (nh, nw) = m2f_preprocess(pil)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(pixel_values=pv[None].to(DEVICE), pixel_mask=pm[None].to(DEVICE))
        masks, labels, scores = m2f_postprocess(out.class_queries_logits[0], out.masks_queries_logits[0],
                                                nh, nw, pil.size[1], pil.size[0])
        for m, l, s in zip(masks, labels, scores):
            if m.any():
                res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(l)],
                            'segmentation': mask_to_rle(m), 'score': float(s)})
    return res

In [ ]:
M2F_BS, M2F_LR, M2F_AMP = 4, 1e-4, USE_AMP     # resep paper: AdamW 1e-4, backbone x0.1, wd 0.05, clip 0.01

def run_m2f():
    run = run_name('mask2former-swinT')
    if already_done(run): return
    seed_everything()
    model = build_m2f()
    dl = torch.utils.data.DataLoader(M2FDataset(TRAIN_JSON, IMG_DIRS['train'], train=True), batch_size=M2F_BS,
                                     shuffle=True, num_workers=2, collate_fn=m2f_collate, pin_memory=True, drop_last=True)
    backbone = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' in n]
    others   = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' not in n]
    opt = torch.optim.AdamW([{'params': backbone, 'lr': M2F_LR * 0.1}, {'params': others, 'lr': M2F_LR}], weight_decay=0.05)
    val_fn = lambda m: coco_eval_segm(VAL_JSON, predict_m2f(m, VAL_JSON, IMG_DIRS['valid']), verbose=False)
    with GPUTimer(run, 'Mask2Former Swin-T', note=f"frac={TRAIN_FRACTION}, amp={M2F_AMP}"):
        best = train_loop(run, model, dl, m2f_step, opt, val_fn, amp=M2F_AMP, clip=0.01)
    model.load_state_dict(torch.load(best, map_location=DEVICE))
    preds = predict_m2f(model, TEST_JSON, IMG_DIRS['test'])
    save_run(run, 'Mask2Former Swin-T', 'transformer', 'test', coco_eval_segm(TEST_JSON, preds), preds)

## D1. Muat 3 model (data latih 100%)

In [ ]:
import re
INPUT_ROOT = '/kaggle/input' if IN_KAGGLE else os.path.abspath('eval_input')
def find_one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits: raise FileNotFoundError(f"{what} tidak ditemukan ({pattern}). Cek Add Input.")
    return hits[0]

# folder CPMK3 output training dicari dari results/*__test.json (cara yang sama dengan notebook evaluasi)
try:
    TRAIN_OUT = os.path.dirname(os.path.dirname(find_one(f"{INPUT_ROOT}/**/results/*__test.json", 'output training')))
except FileNotFoundError:
    print('Isi /kaggle/input:', sorted(glob.glob(f"{INPUT_ROOT}/*/*"))[:20])
    print('File .pt yang ditemukan:', sorted(glob.glob(f"{INPUT_ROOT}/**/*.pt", recursive=True))[:20] or '-')
    raise FileNotFoundError("Output notebook training belum ditambahkan. Add Input -> Your Work -> "
                            "notebook Training_CV3 (versi yang berisi folder CPMK3).")
print('output training:', TRAIN_OUT)
DEMO_RUNS = {'YOLO11s-seg': 'yolo11s-seg_f100_s640',
             'Mask R-CNN': 'maskrcnn-r50fpnv2_f100_s640',
             'Mask2Former': 'mask2former-swinT_f100_s640'}

def weight_path(run):
    w = f"{TRAIN_OUT}/runs/{run}"
    for p in [f"{w}/weights/best.pt", f"{w}/best.pt", f"{w}/weights/last.pt", f"{w}/last.pt"]:
        if os.path.exists(p): return p
    raise FileNotFoundError(run)

MODELS = {}
for name, run in DEMO_RUNS.items():
    if run.startswith('yolo'):
        MODELS[name] = weight_path(run)               # predict_yolo menerima path bobot
    else:
        m = build_maskrcnn(pretrained=False) if run.startswith('maskrcnn') else build_m2f(pretrained=False)
        m.load_state_dict(torch.load(weight_path(run), map_location=DEVICE)); m.to(DEVICE).eval()
        MODELS[name] = m
    print('dimuat:', name, '<-', weight_path(run))

def predict_any(name, gt_json, img_dir):
    # fungsi prediksi yang SAMA dengan notebook training/evaluasi
    if name.startswith('YOLO'): return predict_yolo(MODELS[name], gt_json, img_dir)
    if name.startswith('Mask R'): return predict_maskrcnn(MODELS[name], gt_json, img_dir)
    return predict_m2f(MODELS[name], gt_json, img_dir)

## D2. Inferensi foto baru
Isi `DEMO_IMAGE` dengan path foto. Kalau dibiarkan `None`, dipakai foto paling baru di `/kaggle/working` atau `/kaggle/input/**/demo*`.
Foto diperkecil ke sisi terpanjang 800 px (sama dengan test custom) supaya cepat dan tidak OOM.

**Ganti threshold:** ubah `SCORE_THR` lalu jalankan ulang sel kedua saja (prediksi tidak dihitung ulang).

In [ ]:
DEMO_IMAGE = None          # contoh: '/kaggle/working/foto_baru.jpg'
LONG_SIDE  = 800

try:
    import pillow_heif
except ImportError:
    os.system('pip -q install pillow-heif'); import pillow_heif
pillow_heif.register_heif_opener()   # supaya foto iPhone .HEIC bisa dibuka
if DEMO_IMAGE is None:
    cand = [p for ext in ('jpg', 'jpeg', 'png', 'heic', 'JPG', 'JPEG', 'PNG', 'HEIC')
            for p in glob.glob(f"/kaggle/working/*.{ext}") + glob.glob(f"{INPUT_ROOT}/**/demo*.{ext}", recursive=True)]
    DEMO_IMAGE = max(cand, key=os.path.getmtime) if cand else None
assert DEMO_IMAGE, "belum ada foto demo: upload foto ke /kaggle/working atau isi DEMO_IMAGE"

DEMO_DIR = f"{DATA_ROOT}/demo"; os.makedirs(DEMO_DIR, exist_ok=True)
pil = load_image(DEMO_IMAGE)
s = min(1.0, LONG_SIDE / max(pil.size))
if s < 1: pil = pil.resize((round(pil.size[0] * s), round(pil.size[1] * s)), Image.LANCZOS)
pil.save(f"{DEMO_DIR}/demo.jpg", quality=95)
# GT "kosong" berisi 1 gambar -> fungsi predict_* yang sama bisa dipakai tanpa diubah
DEMO_GT = f"{DEMO_DIR}/demo_gt.json"
json.dump({'images': [{'id': 1, 'file_name': 'demo.jpg', 'width': pil.size[0], 'height': pil.size[1]}],
           'annotations': [], 'categories': [cats_by_id[c] for c in CAT_IDS]}, open(DEMO_GT, 'w'))

DEMO_PREDS = {}
for name in MODELS:
    t0 = time.time(); DEMO_PREDS[name] = predict_any(name, DEMO_GT, DEMO_DIR)
    print(f"{name:12s}: {len(DEMO_PREDS[name]):3d} prediksi mentah, {time.time() - t0:.1f} s")

In [ ]:
SCORE_THR = 0.5            # <-- ubah di sini saat diminta (misal 0.3 atau 0.7), lalu jalankan ulang sel ini

fig, axs = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 6))
for ax, (name, preds) in zip(axs, DEMO_PREDS.items()):
    ps = sorted([p for p in preds if p['score'] >= SCORE_THR], key=lambda p: -p['score'])
    draw_instances(pil, [mask_utils.decode(p['segmentation']).astype(bool) for p in ps],
                   [CATID2IDX[p['category_id']] for p in ps], [p['score'] for p in ps],
                   ax=ax, title=f"{name}: {len(ps)} objek (skor ≥ {SCORE_THR})")
    print(f"{name:12s} skor≥{SCORE_THR}:", ', '.join(f"{cats_by_id[p['category_id']]['name']} {p['score']:.2f}" for p in ps[:12]) or '-')
plt.tight_layout(); plt.show()

## D3. AP per kelas di test custom
- AP dihitung dari **semua** prediksi (skor ≥ 0,001), diurutkan dari skor tertinggi. Threshold tampilan (`SCORE_THR`) **tidak** dipakai AP.
- Sel kedua menunjukkan apa yang terjadi kalau prediksi dipotong dulu dengan threshold: AP turun, karena kurva precision–recall terpotong di recall rendah.

In [ ]:
cj = json.load(open(find_one(f"{INPUT_ROOT}/**/test_custom_clean.json", 'test_custom_clean.json')))
CUSTOM_SRC_DIR = os.path.dirname(find_one(f"{INPUT_ROOT}/**/{glob.escape(cj['images'][0]['file_name'])}", 'gambar test custom'))
TARGET_LONG = 800                                    # sama dengan notebook evaluasi
CUSTOM_IMG_DIR = f"{DATA_ROOT}/custom_imgs"; os.makedirs(CUSTOM_IMG_DIR, exist_ok=True)
for im in cj['images']:
    s = min(1.0, TARGET_LONG / max(im['width'], im['height']))
    p = load_image(os.path.join(CUSTOM_SRC_DIR, im['file_name']))
    if s < 1: p = p.resize((round(im['width'] * s), round(im['height'] * s)), Image.LANCZOS)
    p.save(os.path.join(CUSTOM_IMG_DIR, im['file_name']), quality=95)
    for a in cj['annotations']:
        if a['image_id'] == im['id']:
            a['segmentation'] = [[v * s for v in q] for q in a['segmentation']]
            a['bbox'] = [v * s for v in a['bbox']]; a['area'] *= s * s
    im['width'], im['height'] = p.size
out, _ = clean_split(cj)
CUSTOM_GT = f"{CLEAN_DIR}/test_custom.json"; json.dump(out, open(CUSTOM_GT, 'w'))

CUSTOM_PREDS = {}
for name, run in DEMO_RUNS.items():
    saved = glob.glob(f"{INPUT_ROOT}/**/preds/{run}__test_custom.json", recursive=True)
    CUSTOM_PREDS[name] = json.load(open(saved[0])) if saved else predict_any(name, CUSTOM_GT, CUSTOM_IMG_DIR)
    print(name, '<-', 'prediksi tersimpan' if saved else 'inferensi ulang')

METRICS = {name: coco_eval_segm(CUSTOM_GT, p, verbose=False) for name, p in CUSTOM_PREDS.items()}
n_inst = pd.Series([cats_by_id[a['category_id']]['name'] for a in out['annotations']]).value_counts()
per_cls = pd.DataFrame({n: m['per_class'] for n, m in METRICS.items()}).dropna(how='all')
per_cls.insert(0, 'jumlah_instance', n_inst.reindex(per_cls.index).fillna(0).astype(int))
print({n: round(m['AP'], 4) for n, m in METRICS.items()})
KELAS = None               # <-- isi nama kelas yang diminta, misal 'chair'
display(per_cls.loc[[KELAS]] if KELAS else per_cls.sort_values('jumlah_instance', ascending=False).round(3))

In [ ]:
rows = []
for thr in [0.001, 0.3, 0.5, 0.7]:
    for name, p in CUSTOM_PREDS.items():
        m = coco_eval_segm(CUSTOM_GT, [q for q in p if q['score'] >= thr], verbose=False)
        rows.append({'model': name, 'score_thr': thr, 'AP': m['AP'], 'AP50': m['AP50']})
display(pd.DataFrame(rows).pivot(index='score_thr', columns='model', values='AP').round(4))